# Voice dataset prep: pitch/bass adjustment before RVC training

Downloads only one speaker's files from a dataset (e.g. VCTK), pitch-shifts
and adjusts bass via `pedalboard`, and writes the result ready to feed into
an RVC training notebook (e.g. Applio's) as the training dataset -- instead
of the whole 17GB dataset, and instead of round-tripping through a local
machine.

Run this **before** the training notebook, save its output as a Kaggle
Dataset (Kaggle's "New Dataset" from notebook output), then attach that
dataset to the training notebook instead of the raw source dataset.

In [ ]:
!pip install -q pedalboard soundfile huggingface_hub

## Configure

- `REPO_ID`: the Hugging Face dataset repo to pull from
- `SPEAKER_PATTERN`: glob pattern matching only your chosen speaker's files
  -- check the repo's "Files" tab on huggingface.co first, folder layout
  varies between VCTK mirrors (`wav48/p225/*.wav` vs
  `wav48_silence_trimmed/p225/*_mic1.flac`, etc.)
- `PITCH_SEMITONES`: positive = higher, negative = lower
- `BASS_GAIN_DB`: positive = boost bass, negative = cut it

In [ ]:
REPO_ID = "CSTR-Edinburgh/vctk"  # swap for whichever VCTK mirror you're using
SPEAKER_PATTERN = "wav48_silence_trimmed/p225/*"  # confirm this matches the repo's actual layout

PITCH_SEMITONES = -3.0
BASS_GAIN_DB = 4.0

RAW_DIR = "/kaggle/working/raw"
OUTPUT_DIR = "/kaggle/working/adjusted"

## Download only the chosen speaker's files

`allow_patterns` filters at download time -- only files matching
`SPEAKER_PATTERN` are actually fetched, not the whole dataset.

In [ ]:
from huggingface_hub import snapshot_download

downloaded_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=[SPEAKER_PATTERN],
    local_dir=RAW_DIR,
)
print(f"Downloaded to: {downloaded_path}")

## Apply pitch shift + bass adjustment

Same logic as a plain local script would use -- just running as notebook
cells instead, so the adjusted files never have to leave Kaggle before
training.

In [ ]:
import os
from pathlib import Path

import soundfile as sf
from pedalboard import Pedalboard, PitchShift, LowShelfFilter


def build_board(pitch_semitones, bass_gain_db):
    effects = []
    if pitch_semitones:
        effects.append(PitchShift(semitones=pitch_semitones))
    if bass_gain_db:
        effects.append(LowShelfFilter(cutoff_frequency_hz=200.0, gain_db=bass_gain_db))
    return Pedalboard(effects)


def process_file(board, in_path, out_path):
    audio, sample_rate = sf.read(str(in_path), dtype="float32", always_2d=True)
    processed = board(audio.T, sample_rate)  # pedalboard wants (channels, samples)
    sf.write(str(out_path), processed.T, sample_rate)


raw_dir = Path(RAW_DIR)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

audio_files = sorted(list(raw_dir.rglob("*.wav")) + list(raw_dir.rglob("*.flac")))
if not audio_files:
    raise RuntimeError(f"No .wav/.flac files found under {raw_dir} -- check SPEAKER_PATTERN")

board = build_board(PITCH_SEMITONES, BASS_GAIN_DB)

for i, path in enumerate(audio_files, 1):
    out_path = output_dir / path.name
    process_file(board, path, out_path)
    if i % 25 == 0 or i == len(audio_files):
        print(f"[{i}/{len(audio_files)}] processed")

print(f"Done: {len(audio_files)} file(s) written to {output_dir}")

## Next step

Use Kaggle's "Save Version" then create a **New Dataset** from this
notebook's output (`/kaggle/working/adjusted`) -- then attach that dataset
to your Applio training notebook instead of the raw source dataset.